# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 106.6 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <StringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2000 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [6]:
# Task 1
americas = df.loc[df['continent'] == 'Americas'].copy()
two_years = americas.loc[americas['year'].isin([1997, 2007])].copy()

color_map = {1997: '#AABDD6', 2007: '#1A4E8C'}

fig = px.scatter(
    data_frame=two_years,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map=color_map,
    hover_name='country',
    log_x=True,
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'year': 'Year'
    },
    category_orders={'year': [1997, 2007]}
)

fig.update_traces(
    marker=dict(size=10, opacity=0.8, line=dict(width=0.5, color='white')),
    hovertemplate='<b>%{hovertext}</b><br>GDP per capita: $%{x:,.0f}<br>Life expectancy: %{y:.1f} yrs<extra></extra>'
)

import math
for country, ax, ay in [('Haiti', 55, -30), ('United States', -60, 30)]:
    row = two_years.loc[(two_years['country'] == country) & (two_years['year'] == 2007)].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=f'<b>{country}</b>', showarrow=True,
        arrowhead=2, arrowcolor='#555', ax=ax, ay=ay,
        font=dict(size=11, family='Arial', color='#1A4E8C')
    )

fig.update_layout(
    title='In the Americas, wealth and health both improved 1997–2007 — but the gap between nations persists',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#EEEEEE', showgrid=True),
    yaxis=dict(gridcolor='#EEEEEE', showgrid=True, zeroline=False),
    legend=dict(title='Year'),
    margin=dict(l=60, r=40, t=65, b=40),
    height=520
)

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [8]:
# Task 2
import math

df_2007 = df.loc[df['year'] == 2007].copy()

highlight_continent = 'Asia'
color_map = {c: ('#E07B00' if c == highlight_continent else '#DDDDDD')
             for c in df_2007['continent'].unique()}

fig = px.scatter(
    data_frame=df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    color_discrete_map=color_map,
    hover_name='country',
    log_x=True,
    size_max=55,
    opacity=0.75,
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'continent': ''
    },
    custom_data=['country', 'pop']
)

fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>'
                  'GDP per capita: $%{x:,.0f}<br>'
                  'Life expectancy: %{y:.1f} yrs<br>'
                  'Population: %{customdata[1]:,.0f}<extra></extra>',
    showlegend=False
)

fig.update_traces(
    selector=dict(name=highlight_continent),
    marker=dict(opacity=0.85, line=dict(width=0.8, color='white')),
    showlegend=False
)
fig.update_traces(
    selector=lambda t: t.name != highlight_continent,
    marker=dict(opacity=0.4)
)

for country, ax, ay in [('China', 60, -40), ('Japan', -70, -35), ('Afghanistan', 50, 25)]:
    row = df_2007.loc[df_2007['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=f'<b>{country}</b>', showarrow=True,
        arrowhead=2, arrowcolor='#E07B00', ax=ax, ay=ay,
        font=dict(size=11, family='Arial', color='#E07B00')
    )

fig.add_annotation(
    x=math.log10(1800), y=83,
    text='Asia spans the widest range<br>of any continent — from<br>Afghanistan to Japan',
    showarrow=False,
    font=dict(color='#E07B00', size=10, family='Arial'),
    bgcolor='white', bordercolor='#E07B00', borderwidth=1, borderpad=5,
    align='left'
)

fig.update_layout(
    title='Asia defies generalisations — Japan rivals the West while Afghanistan lags far behind',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    xaxis=dict(gridcolor='#EEEEEE', showgrid=True),
    yaxis=dict(gridcolor='#EEEEEE', showgrid=True, zeroline=False),
    margin=dict(l=60, r=40, t=65, b=40),
    height=580,
    width=1100
)

fig.show()